# 프로젝트 : 뉴스 기사 요약해보기


### 준비. 라이브러리 설치 및 임포트

In [1]:
!pip install --upgrade summa   # Step 5의 추출적 요약(원문 문장을 그대로 뽑는 방식)에 쓸 라이브러리
!pip install --upgrade nltk    # 불용어(stopwords) 목록을 가져오기 위한 라이브러리

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for summa: filename=summa-1.2.0-py3-none-any.whl size=54387 sha256=b7633d34a97cea4c34f9a0ffc964c78ad2019f9d5eb4dde992f354df0dd72dce
  Stored in directory: /root/.cache/pip/wheels/75/be/7d/71f9113116fbdaa9ae7287a3017e9f1c6fbe95cadf170edbbd
Successfully built summa
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.2 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1


In [13]:
import os                                   # 파일 경로/환경 관련 기본 기능 (여기선 거의 안 쓰지만 관례상 import)
import re                                   # 정규표현식 — 전처리에서 특수문자/HTML 잔재 등을 패턴으로 지우는 데 사용
import numpy as np                          # 배열 연산, 셔플/슬라이싱 등 수치 계산용
import pandas as pd                         # CSV를 표(DataFrame) 형태로 다루기 위한 라이브러리
import matplotlib.pyplot as plt             # 길이 분포, loss 그래프 등을 그리기 위한 시각화 라이브러리
%matplotlib inline
from collections import Counter             # 단어 등장 횟수를 세는 데 사용 (단어 집합 만들 때 필요)
from bs4 import BeautifulSoup               # HTML 태그(<br/> 등) 제거용
import nltk

# NLTK stopwords 다운로드 시 프록시 관련 보안 오류를 해결하기 위해 환경 변수 설정
os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'

# NLTK 데이터 경로를 명시적으로 설정하여 stopwords 리소스 탐색 오류 방지
nltk.data.path = ['/root/nltk_data'] + nltk.data.path

nltk.download('stopwords')                  # 영어 불용어(the, a, is 같은 의미 적은 단어) 목록 다운로드
from nltk.corpus import stopwords

import torch                                # PyTorch 본체 — 텐서 연산과 GPU 연산의 기반
import torch.nn as nn                       # 신경망 층(Embedding, LSTM, Linear 등)을 만드는 모듈
import torch.nn.functional as F             # softmax처럼 학습 파라미터가 없는 함수형 연산 모음
import torch.optim as optim                 # AdamW 같은 옵티마이저(가중치 업데이트 규칙) 모음
from torch.utils.data import DataLoader, TensorDataset  # 데이터를 배치 단위로 잘라서 공급해주는 도구
from torch.nn.utils.rnn import pad_sequence # 길이가 제각각인 문장들을 한 텐서로 묶기 위해 길이를 맞춰주는 함수

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='bs4')  # BeautifulSoup의 경고 메시지(파서 관련)를 숨김

print("임포트 완료")

임포트 완료


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Step 1. 데이터 수집하기

데이터는 `news_summary_more.csv`(뉴스 헤드라인/본문)를 사용한다. `headlines`가 강의의 `Summary` 역할(짧은 정답 요약), `text`가 강의의 `Text` 역할(원문)이다.

In [3]:
import urllib.request
# urlretrieve: 인터넷의 파일(csv)을 로컬(코랩 환경)에 그대로 내려받는 함수
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary_more.csv",
    filename="news_summary_more.csv"
)
# encoding='iso-8859-1' (Latin-1): 이 csv가 UTF-8이 아닌 다른 방식으로 저장돼 있어서,
# 기본값(UTF-8)으로 읽으면 특수문자가 깨지거나 에러가 날 수 있어 명시적으로 지정
data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')
data = data[['headlines', 'text']]  # 필요한 두 열만 남김 (headlines=정답 요약, text=원문)
print('전체 샘플수 :', len(data))
data.sample(10)  # 무작위로 10개 뽑아서 데이터 형태를 눈으로 확인

전체 샘플수 : 98401


,headlines,text
6060,Get the f**k out: Santa to kids as fire alarm ...,A man dressed as Santa Claus at an event in th...
74171,Lebanese Army raises Spanish flag for Barcelon...,The Lebanese Army raised a Spanish flag alongs...
91548,Immature Kejriwal's blame-game irked people: B...,"Advocate Prashant Bhushan, who co-founded the ..."
95591,Trump declares April National Sexual Assault A...,US President Donald Trump has declared April t...
76118,Anti-tourism marches spreading across Europe,Anti-tourism marches are spreading across Euro...
69664,10 stray dogs maul minor boy to death in Mumbai,A Class 2 student died after being bitten by a...
39095,2 men lynched after being mistaken as kidnappe...,Two men were lynched in Assam's Karbi Anglong ...
17125,Women made to remove mangalsutras at exam cent...,A few married women taking a Telangana State P...
10540,10-year-old deaf and mute girl dies after bein...,A 10-year-old deaf and mute girl died days aft...
67540,"3,000 killed in Syria in the deadliest month o...","At least 3,000 people including 955 civilians ..."


Step 5(Summa 추출적 요약)에서는 정제되지 않은 원문이 필요하다. 정제되기 전에 원본 `text`를 별도 열로 미리 보존해둔다.

In [4]:
data['text_raw'] = data['text']  # Step 5에서 다시 쓸 정제 전 원문
print('=3')

=3


## Step 2. 데이터 전처리하기 (추상적 요약)

강의(9/4)의 전처리 파이프라인을 그대로 가져오되, 아래 세 가지는 이 데이터셋에 맞게 다시 판단했다.
- **불용어 제거 여부**: `headlines`를 몇 개 직접 확인해보니 전치사·관사가 살아있는 완결된 문장 구조였다 (예: "Malaysia bans travel to North Korea over escalating tensions"). 강의의 `Summary`와 같은 이유로 **불용어를 제거하지 않는다**.
- **최대 길이(`text_max_len`, `summary_max_len`)**: 이 데이터의 실제 길이 분포를 다시 확인하고 정한다 (아래에서 직접 확인).
- **단어 집합 크기(`src_vocab_size`, `tar_vocab_size`)**: 이 데이터의 희귀 단어 비율을 다시 확인하고 정한다 (아래에서 직접 확인).

### 중복 샘플과 NULL 값이 존재하는 샘플 제거

In [5]:
# nunique(): 중복을 뺀 "고유한" 값의 개수. 전체 샘플 수와 차이가 나면 중복 기사가 섞여 있다는 뜻
print('text 열에서 중복을 배제한 유일한 샘플의 수 :', data['text'].nunique())
print('headlines 열에서 중복을 배제한 유일한 샘플의 수 :', data['headlines'].nunique())

text 열에서 중복을 배제한 유일한 샘플의 수 : 98360
headlines 열에서 중복을 배제한 유일한 샘플의 수 : 98280


In [6]:
# inplace=True 를 설정하면 DataFrame 타입 값을 return 하지 않고 data 내부를 직접적으로 바꿉니다
data.drop_duplicates(subset=['text'], inplace=True)
print('전체 샘플수 :', len(data))

전체 샘플수 : 98360


In [7]:
print(data.isnull().sum())  # 각 열마다 비어있는(NULL/NaN) 값이 몇 개인지 확인

headlines    0
text         0
text_raw     0
dtype: int64


In [8]:
data.dropna(axis=0, inplace=True)  # NULL 값이 하나라도 있는 행(샘플) 전체를 삭제
print('전체 샘플수 :', len(data))

전체 샘플수 : 98360


### 텍스트 정규화와 불용어 제거

In [9]:
contractions = {"ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not",
                           "didn't": "did not",  "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not",
                           "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
                           "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have","I'm": "I am", "I've": "I have", "i'd": "i would",
                           "i'd've": "i would have", "i'll": "i will",  "i'll've": "i will have","i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would",
                           "it'd've": "it would have", "it'll": "it will", "it'll've": "it will have","it's": "it is", "let's": "let us", "ma'am": "madam",
                           "mayn't": "may not", "might've": "might have","mightn't": "might not","mightn't've": "might not have", "must've": "must have",
                           "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have","o'clock": "of the clock",
                           "oughtn't": "ought not", "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have",
                           "she'd": "she would", "she'd've": "she would have", "she'll": "she will", "she'll've": "she will have", "she's": "she is",
                           "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have","so's": "so as",
                           "this's": "this is","that'd": "that would", "that'd've": "that would have", "that's": "that is", "there'd": "there would",
                           "there'd've": "there would have", "there's": "there is", "here's": "here is","they'd": "they would", "they'd've": "they would have",
                           "they'll": "they will", "they'll've": "they will have", "they're": "they are", "they've": "they have", "to've": "to have",
                           "wasn't": "was not", "we'd": "we would", "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have", "we're": "we are",
                           "we've": "we have", "weren't": "were not", "what'll": "what will", "what'll've": "what will have", "what're": "what are",
                           "what's": "what is", "what've": "what have", "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is",
                           "where've": "where have", "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have",
                           "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not", "won't've": "will not have",
                           "would've": "would have", "wouldn't": "would not", "wouldn't've": "would not have", "y'all": "you all",
                           "y'all'd": "you all would","y'all'd've": "you all would have","y'all're": "you all are","y'all've": "you all have",
                           "you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have",
                           "you're": "you are", "you've": "you have"}

print("정규화 사전의 수: ", len(contractions))

정규화 사전의 수:  120


In [10]:
# 데이터 전처리 함수 (9/4 강의와 동일)
def preprocess_sentence(sentence, remove_stopwords=True):
    sentence = sentence.lower() # 텍스트 소문자화
    sentence = BeautifulSoup(sentence, "lxml").text # <br />, <a href = ...> 등의 html 태그 제거
    sentence = re.sub(r'\([^)]*\)', '', sentence) # 괄호로 닫힌 문자열 (...) 제거
    sentence = re.sub('"','', sentence) # 쌍따옴표 제거
    sentence = ' '.join([contractions[t] if t in contractions else t for t in sentence.split(" ")]) # 약어 정규화
    sentence = re.sub(r"'s\b","", sentence) # 소유격 제거
    sentence = re.sub("[^a-zA-Z]", " ", sentence) # 영어 외 문자는 공백으로 변환
    sentence = re.sub('[m]{2,}', 'mm', sentence) # 반복 글자 정리

    if remove_stopwords:
        tokens = ' '.join(word for word in sentence.split() if not word in stopwords.words('english') if len(word) > 1)
    else:
        tokens = ' '.join(word for word in sentence.split() if len(word) > 1)
    return tokens
print('=3')

=3


In [14]:
# 함수가 잘 동작하는지 실제 샘플로 확인
print("전처리 전 text :", data['text'].iloc[0])
print("전처리 후 text :", preprocess_sentence(data['text'].iloc[0]))
print("전처리 전 headlines :", data['headlines'].iloc[0])
print("전처리 후 headlines :", preprocess_sentence(data['headlines'].iloc[0], False))

전처리 전 text : Saurav Kant, an alumnus of upGrad and IIIT-B's PG Program in Machine learning and Artificial Intelligence, was a Sr Systems Engineer at Infosys with almost 5 years of work experience. The program and upGrad's 360-degree career support helped him transition to a Data Scientist at Tech Mahindra with 90% salary hike. upGrad's Online Power Learning has powered 3 lakh+ careers.
전처리 후 text : saurav kant alumnus upgrad iiit pg program machine learning artificial intelligence sr systems engineer infosys almost years work experience program upgrad degree career support helped transition data scientist tech mahindra salary hike upgrad online power learning powered lakh careers
전처리 전 headlines : upGrad learner switches to career in ML & Al with 90% salary hike
전처리 후 headlines : upgrad learner switches to career in ml al with salary hike


In [15]:
# 전체 text 데이터에 대한 전처리
clean_text = []
for sentence in data['text']:
    clean_text.append(preprocess_sentence(sentence, remove_stopwords=True))

# headlines에는 불용어 제거 X (전치사/관사가 있는 완결된 문장 구조라 판단 — 위 마크다운 참고)
clean_headlines = []
for sentence in data['headlines']:
    clean_headlines.append(preprocess_sentence(sentence, remove_stopwords=False))

print("Text 전처리 후 결과: ", clean_text[:3])
print("headlines 전처리 후 결과: ", clean_headlines[:3])

Text 전처리 후 결과:  ['saurav kant alumnus upgrad iiit pg program machine learning artificial intelligence sr systems engineer infosys almost years work experience program upgrad degree career support helped transition data scientist tech mahindra salary hike upgrad online power learning powered lakh careers', 'kunal shah credit card bill payment platform cred gave users chance win free food swiggy one year pranav kaushik delhi techie bagged reward spending cred coins users get one cred coin per rupee bill paid used avail rewards brands like ixigo bookmyshow ubereats cult fit', 'new zealand defeated india wickets fourth odi hamilton thursday win first match five match odi series india lost international match rohit sharma captaincy consecutive victories dating back march match witnessed india getting seventh lowest total odi cricket history']
headlines 전처리 후 결과:  ['upgrad learner switches to career in ml al with salary hike', 'delhi techie wins free food from swiggy for one year on cred', '